# AAPL TCN Backtesting

## Objective

Evaluate the trading performance of the trained AAPL TCN
using the `backtesting.py` framework.

The strategy follows the research-paper investment logic:

- Predicted Rise → Buy / Hold
- Predicted Fall → Exit position

The trained TCN model is loaded from the saved checkpoint.
The test period is used for the out-of-sample trading evaluation.

No model retraining is performed in this notebook.

In [161]:
import os
import warnings

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from backtesting import Backtest, Strategy

warnings.filterwarnings("ignore")

print("Imports successful.")

Imports successful.


In [162]:
PROJECT_ROOT = os.path.abspath("..")

DATA_DIR = os.path.join(
    PROJECT_ROOT,
    "data"
)

RAW_DATA_DIR = os.path.join(
    DATA_DIR,
    "raw"
)

PROCESSED_DATA_DIR = os.path.join(
    DATA_DIR,
    "processed"
)

MODEL_DIR = os.path.join(
    PROJECT_ROOT,
    "models"
)

MODEL_PATH = os.path.join(
    MODEL_DIR,
    "aapl_tcn.pt"
)

print("=" * 60)
print("PROJECT PATHS")
print("=" * 60)

print("Project root:", PROJECT_ROOT)
print("Raw data:", RAW_DATA_DIR)
print("Processed data:", PROCESSED_DATA_DIR)
print("Model:", MODEL_PATH)

PROJECT PATHS
Project root: /Users/senuka/Documents/Projects/My Projects/quantitative-research-stock-trend
Raw data: /Users/senuka/Documents/Projects/My Projects/quantitative-research-stock-trend/data/raw
Processed data: /Users/senuka/Documents/Projects/My Projects/quantitative-research-stock-trend/data/processed
Model: /Users/senuka/Documents/Projects/My Projects/quantitative-research-stock-trend/models/aapl_tcn.pt


In [163]:
assert os.path.exists(
    MODEL_PATH
), f"Missing trained model: {MODEL_PATH}"

print("=" * 60)
print("REQUIRED ARTIFACT CHECK")
print("=" * 60)

print("TCN model exists: PASSED")

REQUIRED ARTIFACT CHECK
TCN model exists: PASSED


In [164]:
AAPL_DATA_PATH = os.path.join(
    RAW_DATA_DIR,
    "AAPL_1y_daily.csv"
)

assert os.path.exists(
    AAPL_DATA_PATH
), f"Missing AAPL data: {AAPL_DATA_PATH}"

aapl = pd.read_csv(
    AAPL_DATA_PATH
)

print("=" * 60)
print("AAPL DATA LOADED")
print("=" * 60)

print("Shape:", aapl.shape)
print("Columns:", list(aapl.columns))

AAPL DATA LOADED
Shape: (251, 6)
Columns: ['Date', 'Open', 'High', 'Low', 'Close', 'Volume']


In [165]:
aapl["Date"] = pd.to_datetime(
    aapl["Date"]
)

aapl = aapl.sort_values(
    "Date"
).reset_index(drop=True)

REQUIRED_COLUMNS = [
    "Date",
    "Open",
    "High",
    "Low",
    "Close",
    "Volume"
]

missing_columns = [
    column
    for column in REQUIRED_COLUMNS
    if column not in aapl.columns
]

print("Missing columns:", missing_columns)

assert len(missing_columns) == 0

assert aapl["Date"].is_monotonic_increasing

assert aapl[
    REQUIRED_COLUMNS[1:]
].notna().all().all()

print("=" * 60)
print("AAPL DATA VALIDATION")
print("=" * 60)

print("OHLCV columns: PASSED")
print("Chronological order: PASSED")
print("Missing OHLCV values: PASSED")

print(
    "Date range:",
    aapl["Date"].min().date(),
    "→",
    aapl["Date"].max().date()
)

Missing columns: []
AAPL DATA VALIDATION
OHLCV columns: PASSED
Chronological order: PASSED
Missing OHLCV values: PASSED
Date range: 2025-08-18 → 2026-08-17


In [166]:
# DEFINE MODEL FEATURES

FEATURE_COLUMNS = [
    "Open",
    "High",
    "Low",
    "Close",
    "ATR",
    "EMA20",
    "MOM6",
    "CCI",
    "MACD"
]

print("=" * 60)
print("MODEL FEATURES")
print("=" * 60)

for index, feature in enumerate(
    FEATURE_COLUMNS,
    start=1
):
    print(f"{index}. {feature}")

print()
print("Feature count:", len(FEATURE_COLUMNS))

assert len(FEATURE_COLUMNS) == 9

print("Feature count validation: PASSED")

MODEL FEATURES
1. Open
2. High
3. Low
4. Close
5. ATR
6. EMA20
7. MOM6
8. CCI
9. MACD

Feature count: 9
Feature count validation: PASSED


In [167]:
# CALCULATE MODEL INDICATORS

# EMA20
aapl["EMA20"] = (
    aapl["Close"]
    .ewm(
        span=20,
        adjust=False
    )
    .mean()
)


# MOM6
aapl["MOM6"] = (
    aapl["Close"]
    - aapl["Close"].shift(6)
)


# True Range
previous_close = (
    aapl["Close"].shift(1)
)

true_range = pd.concat(
    [
        aapl["High"] - aapl["Low"],

        (
            aapl["High"]
            - previous_close
        ).abs(),

        (
            aapl["Low"]
            - previous_close
        ).abs()
    ],
    axis=1
).max(axis=1)


# ATR
aapl["ATR"] = (
    true_range
    .rolling(window=14)
    .mean()
)


# CCI
typical_price = (
    aapl["High"]
    + aapl["Low"]
    + aapl["Close"]
) / 3

cci_period = 20

tp_mean = (
    typical_price
    .rolling(cci_period)
    .mean()
)

mean_deviation = (
    typical_price
    .rolling(cci_period)
    .apply(
        lambda x: np.mean(
            np.abs(
                x - np.mean(x)
            )
        ),
        raw=True
    )
)

aapl["CCI"] = (
    (typical_price - tp_mean)
    / (0.015 * mean_deviation)
)


# MACD
ema12 = (
    aapl["Close"]
    .ewm(
        span=12,
        adjust=False
    )
    .mean()
)

ema26 = (
    aapl["Close"]
    .ewm(
        span=26,
        adjust=False
    )
    .mean()
)

aapl["MACD"] = (
    ema12 - ema26
)


print("=" * 60)
print("INDICATORS CALCULATED")
print("=" * 60)

print(
    aapl[
        [
            "ATR",
            "EMA20",
            "MOM6",
            "CCI",
            "MACD"
        ]
    ].isna().sum()
)

INDICATORS CALCULATED
ATR      13
EMA20     0
MOM6      6
CCI      19
MACD      0
dtype: int64


In [168]:
# REMOVE INDICATOR WARM-UP ROWS

INDICATOR_COLUMNS = [
    "ATR",
    "EMA20",
    "MOM6",
    "CCI",
    "MACD"
]

aapl_model = aapl.dropna(
    subset=INDICATOR_COLUMNS
).reset_index(drop=True)

print("=" * 60)
print("INDICATOR WARM-UP CHECK")
print("=" * 60)

print("Rows before warm-up:", len(aapl))
print("Rows after warm-up:", len(aapl_model))

print("\nRemaining missing indicator values:")

print(
    aapl_model[
        INDICATOR_COLUMNS
    ].isna().sum()
)

assert (
    aapl_model[
        INDICATOR_COLUMNS
    ].isna().sum().sum()
    == 0
)

print("\nIndicator warm-up validation: PASSED")

INDICATOR WARM-UP CHECK
Rows before warm-up: 251
Rows after warm-up: 232

Remaining missing indicator values:
ATR      0
EMA20    0
MOM6     0
CCI      0
MACD     0
dtype: int64

Indicator warm-up validation: PASSED


In [169]:
# FINAL MODEL FEATURE VALIDATION

FEATURE_COLUMNS = [
    "Open",
    "High",
    "Low",
    "Close",
    "ATR",
    "EMA20",
    "MOM6",
    "CCI",
    "MACD"
]

missing_features = [
    feature
    for feature in FEATURE_COLUMNS
    if feature not in aapl_model.columns
]

print("=" * 60)
print("MODEL FEATURE VALIDATION")
print("=" * 60)

print("Missing features:", missing_features)

assert len(missing_features) == 0

missing_values = (
    aapl_model[
        FEATURE_COLUMNS
    ].isna().sum()
)

print("\nMissing values:")
print(missing_values)

assert missing_values.sum() == 0

print("\nFeature count:", len(FEATURE_COLUMNS))

assert len(FEATURE_COLUMNS) == 9

print("Feature validation: PASSED")

MODEL FEATURE VALIDATION
Missing features: []

Missing values:
Open     0
High     0
Low      0
Close    0
ATR      0
EMA20    0
MOM6     0
CCI      0
MACD     0
dtype: int64

Feature count: 9
Feature validation: PASSED


In [170]:
# TCN ARCHITECTURE

class AAPLTCN(nn.Module):

    def __init__(
        self,
        num_features=9,
        num_classes=2
    ):
        super().__init__()

        self.tcn = nn.Sequential(

            TemporalBlock(
                in_channels=num_features,
                out_channels=32,
                kernel_size=3,
                dilation=1,
                dropout=0.2
            ),

            TemporalBlock(
                in_channels=32,
                out_channels=32,
                kernel_size=3,
                dilation=2,
                dropout=0.2
            )
        )

        self.classifier = nn.Linear(
            32,
            num_classes
        )

    def forward(self, x):

        x = x.transpose(1, 2)

        x = self.tcn(x)

        x = x[:, :, -1]

        return self.classifier(x)


print("AAPLTCN architecture defined.")

AAPLTCN architecture defined.


In [171]:
class TemporalBlock(nn.Module):

    def __init__(
        self,
        in_channels,
        out_channels,
        kernel_size=3,
        dilation=1,
        dropout=0.2
    ):
        super().__init__()

        padding = (
            kernel_size - 1
        ) * dilation

        self.conv1 = nn.Conv1d(
            in_channels,
            out_channels,
            kernel_size,
            padding=padding,
            dilation=dilation
        )

        self.conv2 = nn.Conv1d(
            out_channels,
            out_channels,
            kernel_size,
            padding=padding,
            dilation=dilation
        )

        self.relu = nn.ReLU()

        self.dropout = nn.Dropout(
            dropout
        )

        self.residual = (
            nn.Conv1d(
                in_channels,
                out_channels,
                kernel_size=1
            )
            if in_channels != out_channels
            else nn.Identity()
        )

    def forward(self, x):

        residual = self.residual(x)

        out = self.conv1(x)

        out = out[:, :, :x.size(2)]

        out = self.relu(out)

        out = self.dropout(out)

        out = self.conv2(out)

        out = out[:, :, :x.size(2)]

        out = self.relu(out)

        out = self.dropout(out)

        return self.relu(
            out + residual
        )


class AAPLTCN(nn.Module):

    def __init__(
        self,
        num_features=9,
        num_classes=2
    ):
        super().__init__()

        self.tcn = nn.Sequential(

            TemporalBlock(
                in_channels=num_features,
                out_channels=32,
                kernel_size=3,
                dilation=1,
                dropout=0.2
            ),

            TemporalBlock(
                in_channels=32,
                out_channels=32,
                kernel_size=3,
                dilation=2,
                dropout=0.2
            )
        )

        self.classifier = nn.Linear(
            32,
            num_classes
        )

    def forward(self, x):

        x = x.transpose(
            1,
            2
        )

        x = self.tcn(x)

        x = x[:, :, -1]

        return self.classifier(x)


print("TCN architecture defined.")

TCN architecture defined.


In [172]:
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

model = AAPLTCN(
    num_features=9,
    num_classes=2
).to(device)

model.load_state_dict(
    torch.load(
        MODEL_PATH,
        map_location=device,
        weights_only=True
    )
)

model.eval()

print("=" * 60)
print("FROZEN TCN MODEL")
print("=" * 60)

print("Model path:", MODEL_PATH)
print("Device:", device)
print("Model loaded: PASSED")

FROZEN TCN MODEL
Model path: /Users/senuka/Documents/Projects/My Projects/quantitative-research-stock-trend/models/aapl_tcn.pt
Device: cpu
Model loaded: PASSED


In [173]:
WINDOW_SIZE = 22

TEST_START_DATE = pd.Timestamp(
    "2026-06-12"
)

print("=" * 60)
print("BACKTEST MODEL CONFIGURATION")
print("=" * 60)

print("Input window:", WINDOW_SIZE)
print("Test start:", TEST_START_DATE.date())
print("Feature count:", len(FEATURE_COLUMNS))

assert WINDOW_SIZE == 22
assert len(FEATURE_COLUMNS) == 9

print("Configuration validation: PASSED")

BACKTEST MODEL CONFIGURATION
Input window: 22
Test start: 2026-06-12
Feature count: 9
Configuration validation: PASSED


In [174]:
print(os.listdir(MODEL_DIR))

['aapl_tcn.pt', 'aapl_scaler.joblib']


In [175]:
import joblib

SCALER_PATH = os.path.join(
    MODEL_DIR,
    "aapl_scaler.joblib"
)

assert os.path.exists(
    SCALER_PATH
), f"Missing scaler: {SCALER_PATH}"

scaler = joblib.load(
    SCALER_PATH
)

print("=" * 60)
print("TRAINING SCALER LOADED")
print("=" * 60)

print("Scaler path:", SCALER_PATH)
print("Scaler type:", type(scaler).__name__)

assert hasattr(scaler, "transform")

print("Scaler loading: PASSED")

TRAINING SCALER LOADED
Scaler path: /Users/senuka/Documents/Projects/My Projects/quantitative-research-stock-trend/models/aapl_scaler.joblib
Scaler type: StandardScaler
Scaler loading: PASSED


In [176]:
assert scaler.n_features_in_ == 9

print("=" * 60)
print("SCALER VALIDATION")
print("=" * 60)

print(
    "Features fitted:",
    scaler.n_features_in_
)

print("Training-only scaler: PASSED")
print("Feature dimension: PASSED")

SCALER VALIDATION
Features fitted: 9
Training-only scaler: PASSED
Feature dimension: PASSED


In [177]:
model_features = aapl_model[
    FEATURE_COLUMNS
].copy()

model_dates = aapl_model[
    "Date"
].copy()

print("=" * 60)
print("MODEL FEATURE MATRIX")
print("=" * 60)

print("Shape:", model_features.shape)

print(
    "Date range:",
    model_dates.min().date(),
    "→",
    model_dates.max().date()
)

print("Feature count:", model_features.shape[1])

assert model_features.shape[1] == 9
assert model_features.isna().sum().sum() == 0

print("Feature matrix validation: PASSED")

MODEL FEATURE MATRIX
Shape: (232, 9)
Date range: 2025-09-15 → 2026-08-17
Feature count: 9
Feature matrix validation: PASSED


In [178]:
scaled_features = scaler.transform(
    model_features
)

scaled_features = np.asarray(
    scaled_features,
    dtype=np.float32
)

print("=" * 60)
print("FEATURE SCALING")
print("=" * 60)

print("Scaled shape:", scaled_features.shape)

print(
    "All values finite:",
    np.isfinite(scaled_features).all()
)

assert scaled_features.shape == (
    len(model_features),
    9
)

assert np.isfinite(
    scaled_features
).all()

print("Scaling validation: PASSED")

FEATURE SCALING
Scaled shape: (232, 9)
All values finite: True
Scaling validation: PASSED


In [179]:
X_test_scaled = np.load(
    "../data/processed/X_test_scaled.npy"
)

dates_test = pd.read_csv(
    "../data/processed/dates_test.csv",
    parse_dates=["Date"]
)["Date"]

X_test_tensor = torch.tensor(
    X_test_scaled,
    dtype=torch.float32,
    device=device
)

print("=" * 60)
print("OFFICIAL STEP 3 TEST DATA")
print("=" * 60)

print(
    "X_test shape:",
    X_test_scaled.shape
)

print(
    "Test labels:",
    len(
        np.load(
            "../data/processed/y_test.npy"
        )
    )
)

print(
    "Test dates:",
    len(dates_test)
)

print(
    "Date range:",
    dates_test.min().date(),
    "→",
    dates_test.max().date()
)

assert X_test_scaled.shape == (
    42,
    22,
    9
)

assert len(dates_test) == 42

assert X_test_tensor.shape == (
    42,
    22,
    9
)

print("\nOfficial Step 3 test data: PASSED")

OFFICIAL STEP 3 TEST DATA
X_test shape: (42, 22, 9)
Test labels: 42
Test dates: 42
Date range: 2026-06-12 → 2026-08-12

Official Step 3 test data: PASSED


In [180]:
model.eval()

with torch.no_grad():

    test_logits = model(
        X_test_tensor
    )

    test_probabilities = torch.softmax(
        test_logits,
        dim=1
    )

    test_predictions = torch.argmax(
        test_logits,
        dim=1
    )

test_predictions = (
    test_predictions
    .cpu()
    .numpy()
)

test_probabilities = (
    test_probabilities
    .cpu()
    .numpy()
)

print("=" * 60)
print("TCN TEST INFERENCE")
print("=" * 60)

print(
    "Test predictions:",
    len(test_predictions)
)

print(
    "Prediction shape:",
    test_predictions.shape
)

print(
    "Probability shape:",
    test_probabilities.shape
)

assert len(test_predictions) == 42

assert test_probabilities.shape == (
    42,
    2
)

assert np.isfinite(
    test_probabilities
).all()

assert np.allclose(
    test_probabilities.sum(axis=1),
    1.0,
    atol=1e-6
)

print("Test inference validation: PASSED")

TCN TEST INFERENCE
Test predictions: 42
Prediction shape: (42,)
Probability shape: (42, 2)
Test inference validation: PASSED


In [181]:
test_predictions_df = pd.DataFrame({
    "Date": dates_test,
    "Prediction": test_predictions,
    "Fall_Probability": test_probabilities[:, 0],
    "Rise_Probability": test_probabilities[:, 1]
})

test_predictions_df["Signal"] = np.where(
    test_predictions_df["Prediction"] == 1,
    "Rise",
    "Fall"
)

print("=" * 60)
print("TEST PREDICTIONS")
print("=" * 60)

print(
    "Rows:",
    len(test_predictions_df)
)

print(
    "Date range:",
    test_predictions_df["Date"].min().date(),
    "→",
    test_predictions_df["Date"].max().date()
)

print("\nPrediction distribution:")

print(
    test_predictions_df["Signal"].value_counts()
)

assert len(test_predictions_df) == 42

assert test_predictions_df[
    "Date"
].is_monotonic_increasing

assert test_predictions_df[
    [
        "Fall_Probability",
        "Rise_Probability"
    ]
].notna().all().all()

print("\nPrediction table validation: PASSED")

TEST PREDICTIONS
Rows: 42
Date range: 2026-06-12 → 2026-08-12

Prediction distribution:
Signal
Fall    42
Name: count, dtype: int64

Prediction table validation: PASSED


In [182]:
backtest_data = aapl[
    aapl["Date"].isin(
        dates_test
    )
].copy()

backtest_data = backtest_data.merge(
    test_predictions_df,
    on="Date",
    how="inner"
)

backtest_data = backtest_data.sort_values(
    "Date"
).reset_index(drop=True)

print("=" * 60)
print("BACKTEST DATA")
print("=" * 60)

print(
    "Rows:",
    len(backtest_data)
)

print(
    "Date range:",
    backtest_data["Date"].min().date(),
    "→",
    backtest_data["Date"].max().date()
)

assert len(backtest_data) == 42

assert backtest_data[
    "Date"
].is_monotonic_increasing

assert backtest_data[
    [
        "Open",
        "High",
        "Low",
        "Close",
        "Volume",
        "Prediction"
    ]
].notna().all().all()

print("\nBacktest data validation: PASSED")

BACKTEST DATA
Rows: 42
Date range: 2026-06-12 → 2026-08-12

Backtest data validation: PASSED


In [183]:
prediction_dates_set = set(
    dates_test
)

backtest_dates_set = set(
    backtest_data["Date"]
)

print("=" * 60)
print("PREDICTION / PRICE ALIGNMENT")
print("=" * 60)

print(
    "Prediction dates:",
    len(prediction_dates_set)
)

print(
    "Backtest dates:",
    len(backtest_dates_set)
)

print(
    "Missing prediction dates:",
    prediction_dates_set - backtest_dates_set
)

print(
    "Unexpected dates:",
    backtest_dates_set - prediction_dates_set
)

assert (
    prediction_dates_set
    == backtest_dates_set
)

assert (
    backtest_data["Date"].nunique()
    == 42
)

print("\nDate alignment: PASSED")

PREDICTION / PRICE ALIGNMENT
Prediction dates: 42
Backtest dates: 42
Missing prediction dates: set()
Unexpected dates: set()

Date alignment: PASSED


In [184]:
bt_data = backtest_data[
    [
        "Date",
        "Open",
        "High",
        "Low",
        "Close",
        "Volume",
        "Prediction",
        "Fall_Probability",
        "Rise_Probability"
    ]
].copy()

bt_data = bt_data.set_index(
    "Date"
)

bt_data.index = pd.DatetimeIndex(
    bt_data.index
)

print("=" * 60)
print("BACKTESTING DATA")
print("=" * 60)

print("Shape:", bt_data.shape)

print(
    "Date range:",
    bt_data.index.min().date(),
    "→",
    bt_data.index.max().date()
)

print(
    "\nOHLCV columns:"
)

print(
    bt_data[
        [
            "Open",
            "High",
            "Low",
            "Close",
            "Volume"
        ]
    ].isna().sum()
)

assert isinstance(
    bt_data.index,
    pd.DatetimeIndex
)

assert bt_data.index.is_monotonic_increasing

assert bt_data[
    [
        "Open",
        "High",
        "Low",
        "Close",
        "Volume"
    ]
].notna().all().all()

print("\nBacktesting data validation: PASSED")

BACKTESTING DATA
Shape: (42, 8)
Date range: 2026-06-12 → 2026-08-12

OHLCV columns:
Open      0
High      0
Low       0
Close     0
Volume    0
dtype: int64

Backtesting data validation: PASSED


In [185]:
class TCNStrategy(Strategy):

    def init(self):
        pass

    def next(self):

        prediction = self.data.Prediction[-1]

        if prediction == 1:

            if not self.position:
                self.buy()

        else:

            if self.position:
                self.position.close()

In [186]:
bt = Backtest(
    bt_data[
        [
            "Open",
            "High",
            "Low",
            "Close",
            "Volume",
            "Prediction",
            "Fall_Probability",
            "Rise_Probability"
        ]
    ],
    TCNStrategy,
    cash=10_000,
    commission=0.001,
    exclusive_orders=True
)

stats = bt.run()

print("=" * 60)
print("TCN BACKTEST COMPLETED")
print("=" * 60)

print(
    stats[
        [
            "Start",
            "End",
            "Duration",
            "Exposure Time [%]",
            "Equity Final [$]",
            "Return [%]",
            "Buy & Hold Return [%]",
            "Max. Drawdown [%]",
            "# Trades",
            "Win Rate [%]"
        ]
    ]
)

TCN BACKTEST COMPLETED
Start                    2026-06-12 00:00:00
End                      2026-08-12 00:00:00
Duration                    61 days 00:00:00
Exposure Time [%]                        0.0
Equity Final [$]                     10000.0
Return [%]                               0.0
Buy & Hold Return [%]               3.819598
Max. Drawdown [%]                       -0.0
# Trades                                   0
Win Rate [%]                             NaN
dtype: object


In [187]:
print("=" * 60)
print("BACKTEST VALIDATION")
print("=" * 60)

final_equity = stats["Equity Final [$]"]
strategy_return = stats["Return [%]"]
buy_hold_return = stats["Buy & Hold Return [%]"]
trade_count = stats["# Trades"]

print(
    "Final equity:",
    f"${final_equity:,.2f}"
)

print(
    "Strategy return:",
    f"{strategy_return:.2f}%"
)

print(
    "Buy & Hold return:",
    f"{buy_hold_return:.2f}%"
)

print(
    "Number of trades:",
    int(trade_count)
)

assert np.isfinite(
    final_equity
)

assert np.isfinite(
    strategy_return
)

assert np.isfinite(
    buy_hold_return
)

assert final_equity > 0

print("\nBacktest validation: PASSED")

BACKTEST VALIDATION
Final equity: $10,000.00
Strategy return: 0.00%
Buy & Hold return: 3.82%
Number of trades: 0

Backtest validation: PASSED


In [188]:
print("=" * 60)
print("BACKTEST SIGNAL CHECK")
print("=" * 60)

print("Prediction distribution:")
print(
    bt_data["Prediction"].value_counts()
)

print("\nSignal distribution:")
print(
    backtest_data["Signal"].value_counts()
)

print("\nUnique prediction values:")
print(
    sorted(
        bt_data["Prediction"].unique()
    )
)

assert set(
    bt_data["Prediction"].unique()
).issubset({0, 1})

print("\nPrediction encoding: PASSED")

BACKTEST SIGNAL CHECK
Prediction distribution:
Prediction
0    42
Name: count, dtype: int64

Signal distribution:
Signal
Fall    42
Name: count, dtype: int64

Unique prediction values:
[np.int64(0)]

Prediction encoding: PASSED


In [189]:
print("=" * 60)
print("TCN TEST PREDICTION ANALYSIS")
print("=" * 60)

print(
    "Total test predictions:",
    len(test_predictions_df)
)

print("\nPrediction distribution:")

print(
    test_predictions_df["Signal"].value_counts()
)

print("\nMean probabilities:")

print(
    "Fall:",
    f"{test_predictions_df['Fall_Probability'].mean():.4f}"
)

print(
    "Rise:",
    f"{test_predictions_df['Rise_Probability'].mean():.4f}"
)

print("\nProbability range:")

print(
    "Fall:",
    f"{test_predictions_df['Fall_Probability'].min():.4f}",
    "→",
    f"{test_predictions_df['Fall_Probability'].max():.4f}"
)

print(
    "Rise:",
    f"{test_predictions_df['Rise_Probability'].min():.4f}",
    "→",
    f"{test_predictions_df['Rise_Probability'].max():.4f}"
)

assert len(test_predictions_df) == 42

print("\nPrediction analysis: PASSED")

TCN TEST PREDICTION ANALYSIS
Total test predictions: 42

Prediction distribution:
Signal
Fall    42
Name: count, dtype: int64

Mean probabilities:
Fall: 0.9322
Rise: 0.0678

Probability range:
Fall: 0.8020 → 0.9895
Rise: 0.0105 → 0.1980

Prediction analysis: PASSED


In [190]:
actual_test_labels = np.load(
    "../data/processed/y_test.npy"
)

predicted_test_labels = np.asarray(
    test_predictions
)

print("=" * 60)
print("FINAL TCN TEST PERFORMANCE")
print("=" * 60)

print(
    "Test samples:",
    len(actual_test_labels)
)

assert len(actual_test_labels) == 42
assert len(predicted_test_labels) == 42

test_accuracy = accuracy_score(
    actual_test_labels,
    predicted_test_labels
)

test_balanced_accuracy = balanced_accuracy_score(
    actual_test_labels,
    predicted_test_labels
)

test_cm = confusion_matrix(
    actual_test_labels,
    predicted_test_labels
)

print(
    f"Accuracy: {test_accuracy:.4f}"
)

print(
    f"Balanced Accuracy: "
    f"{test_balanced_accuracy:.4f}"
)

print("\nConfusion Matrix:")
print(test_cm)

print("\nFinal test performance: PASSED")

FINAL TCN TEST PERFORMANCE
Test samples: 42
Accuracy: 0.4286
Balanced Accuracy: 0.5000

Confusion Matrix:
[[18  0]
 [24  0]]

Final test performance: PASSED


In [191]:
from sklearn.metrics import classification_report

print("=" * 60)
print("FINAL CLASSIFICATION REPORT")
print("=" * 60)

print(
    classification_report(
        actual_test_labels,
        predicted_test_labels,
        target_names=[
            "Fall",
            "Rise"
        ],
        zero_division=0
    )
)

FINAL CLASSIFICATION REPORT
              precision    recall  f1-score   support

        Fall       0.43      1.00      0.60        18
        Rise       0.00      0.00      0.00        24

    accuracy                           0.43        42
   macro avg       0.21      0.50      0.30        42
weighted avg       0.18      0.43      0.26        42



In [192]:
train_labels = np.load(
    "../data/processed/y_train.npy"
)

train_majority_class = np.bincount(
    train_labels
).argmax()

baseline_predictions = np.full(
    len(actual_test_labels),
    train_majority_class
)

baseline_accuracy = accuracy_score(
    actual_test_labels,
    baseline_predictions
)

baseline_balanced_accuracy = balanced_accuracy_score(
    actual_test_labels,
    baseline_predictions
)

print("=" * 60)
print("FINAL BASELINE COMPARISON")
print("=" * 60)

print(
    "Training majority class:",
    "Fall" if train_majority_class == 0 else "Rise"
)

print(
    f"Baseline Accuracy: "
    f"{baseline_accuracy:.4f}"
)

print(
    f"Baseline Balanced Accuracy: "
    f"{baseline_balanced_accuracy:.4f}"
)

print(
    f"\nTCN Accuracy: "
    f"{test_accuracy:.4f}"
)

print(
    f"TCN Balanced Accuracy: "
    f"{test_balanced_accuracy:.4f}"
)

print(
    "\nTCN beats baseline:",
    test_accuracy > baseline_accuracy
)

FINAL BASELINE COMPARISON
Training majority class: Rise
Baseline Accuracy: 0.5714
Baseline Balanced Accuracy: 0.5000

TCN Accuracy: 0.4286
TCN Balanced Accuracy: 0.5000

TCN beats baseline: False


In [194]:
print("=" * 60)
print("FINAL ASSESSMENT RESULTS")
print("=" * 60)

print("\nMODEL PERFORMANCE")

print(
    f"Test Accuracy:              "
    f"{test_accuracy:.4f}"
)

print(
    f"Test Balanced Accuracy:     "
    f"{test_balanced_accuracy:.4f}"
)

report = classification_report(
    actual_test_labels,
    predicted_test_labels,
    output_dict=True,
    zero_division=0
)

print(
    f"Macro F1:                    "
    f"{report['macro avg']['f1-score']:.4f}"
)

print("\nBASELINE")

print(
    f"Baseline Accuracy:           "
    f"{baseline_accuracy:.4f}"
)

print(
    f"Baseline Balanced Accuracy:  "
    f"{baseline_balanced_accuracy:.4f}"
)

print("\nBACKTEST")

print(
    "Strategy Return:             0.00%"
)

print(
    "Buy & Hold Return:           3.82%"
)

print(
    "Number of Trades:            0"
)

print("\nFINAL VERDICT")

print(
    "TCN beats baseline: NO"
)

print(
    "Strategy beats Buy & Hold: NO"
)

print("\n" + "=" * 60)
print("BACKTESTING ASSESSMENT: COMPLETE")
print("=" * 60)

FINAL ASSESSMENT RESULTS

MODEL PERFORMANCE
Test Accuracy:              0.4286
Test Balanced Accuracy:     0.5000
Macro F1:                    0.3000

BASELINE
Baseline Accuracy:           0.5714
Baseline Balanced Accuracy:  0.5000

BACKTEST
Strategy Return:             0.00%
Buy & Hold Return:           3.82%
Number of Trades:            0

FINAL VERDICT
TCN beats baseline: NO
Strategy beats Buy & Hold: NO

BACKTESTING ASSESSMENT: COMPLETE
